In [1]:
import glob
import json
import os
import uuid

from dotenv import load_dotenv
import numpy as np
from PIL import Image

import torch

import matplotlib.pyplot as plt

from koger_detection.obj_det.predictors import Predictor
from koger_detection.obj_det.mydatasets import ImageDataset
from koger_detection.obj_det.engine import collate_fn, worker_init_fn

INFO:albumentations.check_version:A new version of Albumentations is available: 1.4.20 (you have 1.4.11). Upgrade using: pip install --upgrade albumentations


In [2]:
# Assumes we have a local .env file that stores things like ROOT
load_dotenv()

images_folder = "/mnt/h/Pronghorn Vertical Imagery/2024/PR527"

output_folder = os.path.join("/home/koger/pronghorn-processing", model_name)
os.makedirs(output_folder, exist_ok=True)

research_project = "pronghorn-survey"

In [3]:
model_name = "10-25-2024-16-50-17"

model_path = os.environ.get("MODEL_PATH")
model_folder = os.path.join(model_path, research_project, "runs", model_name)
model_cfg_file = os.path.join(model_folder, "cfg.json")
model_weights_file = os.path.join(model_folder, "final_model.pth")

image_files = sorted(glob.glob(os.path.join(images_folder, f"*.[jJ][pP][gG]")))
print(f"{len(image_files)} files found.")

5722 files found.


In [5]:
with open(model_cfg_file) as f:
    cfg = json.load(f)
    model_cfg = cfg['model']
model_cfg['fixed_size'] = [8256, 5504]
model_cfg['model_weights_pth'] = model_weights_file

In [8]:
rgb = True

image_dataset = ImageDataset(image_files, rgb=rgb)
dataloader = torch.utils.data.DataLoader(
            image_dataset, batch_size=1, shuffle=False, 
            num_workers=4)
# TODO!!!!!! rgb being true here actually means it is converted to bgr
# I think should actually stay as rgb
predictor = Predictor(model_cfg, invert_color_channel=False)

In [9]:
for ind, image in enumerate(dataloader):

    if ind % 300 == 0:
        print(ind)

    res = predictor(image['image'][0])
    
    boxes = res['boxes'].to('cpu').numpy().astype(np.uint32)
    scores = res['scores'].to('cpu').numpy()
    labels = res['labels'].to('cpu').detach().numpy()

    image_name = os.path.splitext(os.path.basename(image['filename'][0]))[0]

    np.save(os.path.join(output_folder, f"{image_name}_boxes.npy"), boxes)
    np.save(os.path.join(output_folder, f"{image_name}_scores.npy"), scores)
    np.save(os.path.join(output_folder, f"{image_name}_labels.npy"), labels)


0
300
600
900
1200
1500
1800
2100
2400
2700
3000
3300
3600
3900
4200
4500
4800
5100
5400
5700
